# ICU Deterioration — Phase 1: EDA
Run after placing your MIMIC CSV in `data/raw/`.

In [ ]:
import sys
sys.path.append('../src/cloud')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from data_pipeline import load_raw, clean, engineer_features, select_and_impute, VITAL_COLS, TARGET_COL

df_raw = load_raw()
df     = clean(df_raw)
df     = engineer_features(df)
print(df.shape)
df.head(3)

In [ ]:
# ── Target distribution
counts = df[TARGET_COL].value_counts()
print('Class distribution:')
print(counts)
print(f'\nPositive rate: {counts[1]/len(df)*100:.1f}%')

fig, ax = plt.subplots(figsize=(4,3))
ax.bar(['Survived (0)', 'Died in ICU (1)'], counts.values, color=['#5DCAA5','#D85A30'])
ax.set_title('ICU mortality distribution')
ax.set_ylabel('Patients')
plt.tight_layout()
plt.show()

In [ ]:
# ── Missing-value heatmap ────────────────────────────────────────────────
vital_cols_present = [c for c in VITAL_COLS if c in df.columns]
missing = df[vital_cols_present].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 4))
missing.plot.bar(ax=ax, color='#7F77DD')
ax.set_ylabel('% missing')
ax.set_title('Missing values per vital-sign column')
ax.axhline(50, color='red', linestyle='--', linewidth=0.8, label='50% threshold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Vital-sign distributions by outcome ─────────────────────────────────
plot_cols = ['heartrate_mean','sysbp_mean','resprate_mean','spo2_mean','tempc_mean','news2_score']
plot_cols = [c for c in plot_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, plot_cols):
    for label, color in [(0,'#5DCAA5'),(1,'#D85A30')]:
        vals = df[df[TARGET_COL]==label][col].dropna()
        ax.hist(vals, bins=40, alpha=0.55, color=color,
                label=f'ICU death={label} (n={len(vals)})', density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Vital distributions by ICU mortality', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────
X, y, features = select_and_impute(df)
corr = X.corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.columns, fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── NEWS-2 score vs mortality rate ───────────────────────────────────────
if 'news2_score' in df.columns:
    news_mort = df.groupby('news2_score')[TARGET_COL].mean() * 100
    fig, ax = plt.subplots(figsize=(8,4))
    news_mort.plot.bar(ax=ax, color='#D85A30')
    ax.set_xlabel('NEWS-2 score')
    ax.set_ylabel('ICU mortality rate (%)')
    ax.set_title('NEWS-2 score vs ICU mortality')
    plt.tight_layout()
    plt.show()
    print('Higher NEWS-2 → higher mortality, as expected.')

In [ ]:
# ── Run full pipeline and confirm saved files ─────────────────────────────
from data_pipeline import run_pipeline
summary = run_pipeline()
print('\nPipeline summary:')
for k,v in summary.items():
    if k != 'feature_names':
        print(f'  {k}: {v}')
print(f'  features: {summary["features"]} columns')